# HW – Part 1: Regression with PyTorch → ONNX → GitHub Pages
**Dataset:** Concrete Compressive Strength (Yeh, 1998; UCI Machine Learning Repository)

**Goal:** predict the compressive strength of concrete (MPa) from its mix recipe (7 ingredients, kg/m³) and its age (days).

Pipeline: read data → EDA → cleaning & feature processing → train/val/test split → 5 PyTorch regressors (Linear, MLP, Deep NN, F1+F2, ResidualNet) trained with the same loop → evaluate with R², MAE, RMSE → analysis → export the best model to ONNX for the web app.

Following the approach in class, the **standardization is stored inside each model** (`x_means`, `x_deviations`, `y_means`, `y_deviations`), so the exported ONNX file takes raw inputs and returns MPa directly.

In [ ]:
import os, json, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.set_printoptions(suppress=True)
torch.set_printoptions(sci_mode=False)

OUT_FIG  = "outputs/figures"
OUT_RES  = "outputs/results"
OUT_WEB  = "outputs/docs/regression"          # copy this folder into your GitHub Pages repo
for d in [OUT_FIG, OUT_RES, OUT_WEB]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Read the data

In [ ]:
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/concrete.csv"
LOCAL = "concrete.csv"
df_raw = pd.read_csv(LOCAL) if os.path.exists(LOCAL) else pd.read_csv(URL)
print(df_raw.shape)
df_raw.head()

In [ ]:
FEATURES = ["cement", "slag", "ash", "water", "superplastic", "coarseagg", "fineagg", "age"]
TARGET   = "strength"
UNITS    = {"cement": "kg/m³", "slag": "kg/m³", "ash": "kg/m³", "water": "kg/m³",
            "superplastic": "kg/m³", "coarseagg": "kg/m³", "fineagg": "kg/m³",
            "age": "days", "strength": "MPa"}
df_raw.info()
df_raw.describe().T

## 2. Exploratory data analysis

In [ ]:
print("Missing values:", int(df_raw.isna().sum().sum()))
print("Duplicate rows:", int(df_raw.duplicated().sum()))

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.ravel(), FEATURES + [TARGET]):
    sns.histplot(df_raw[col], bins=30, ax=ax, color="#3b6ea5")
    ax.set_title(f"{col} ({UNITS[col]})")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_distributions.png", dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_raw.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Concrete – Pearson correlation matrix")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_correlation.png", dpi=150); plt.show()
print(corr[TARGET].drop(TARGET).sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(df_raw["age"], df_raw[TARGET], s=8, alpha=.5)
axes[0].set_xlabel("age (days)"); axes[0].set_ylabel("strength (MPa)"); axes[0].set_title("Raw age: strongly non-linear")
axes[1].scatter(np.log1p(df_raw["age"]), df_raw[TARGET], s=8, alpha=.5, color="#c0504d")
axes[1].set_xlabel("log(1 + age)"); axes[1].set_title("log(1+age): close to linear")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_age_log.png", dpi=150); plt.show()

**Observations**
* No missing values, but there are duplicated rows → drop them so identical rows cannot appear in both train and test (data leakage).
* `cement`, `superplastic` and `age` correlate positively with strength; `water` negatively (the classic water/cement ratio effect).
* `age` is heavily right-skewed (1–365 days) and its effect saturates (concrete gains most of its strength in the first 28 days). A `log(1+age)` transform makes the relation almost linear. The transform is done **inside the model** so the web app can accept age in days.

## 3. Processing and split (70 % train / 15 % validation / 15 % test)

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

X = df[FEATURES].values.astype(np.float32)
y = df[[TARGET]].values.astype(np.float32)

SPLIT_SEED = 42
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SPLIT_SEED)
X_val,   X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SPLIT_SEED)
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)

AGE_IDX = FEATURES.index("age")
assert AGE_IDX == len(FEATURES) - 1, "age is expected to be the last column"

def engineer(x, log_age=True):
    """Feature engineering applied identically in training and inside the exported model:
    replace age (last column) by log(1 + age)."""
    if not log_age:
        return x
    return torch.cat([x[:, :AGE_IDX], torch.log1p(x[:, AGE_IDX:])], dim=1)

def scaling_stats(log_age=True):
    """Means/std computed on the TRAINING set only (no leakage)."""
    xt = engineer(torch.tensor(X_train), log_age)
    yt = torch.tensor(y_train)
    return xt.mean(0), xt.std(0), yt.mean(0), yt.std(0)

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
train_ds = TensorDataset(to_t(X_train), to_t(y_train))

## 4. Models (PyTorch, as in class)
All models share a base class that (1) applies the feature engineering, (2) standardizes x with training statistics, (3) predicts a **scaled** y and (4) returns `(pred_descaled, pred_scaled)` – the same interface used in the class examples.

| Model | Idea |
|---|---|
| **Linear** | `nn.Linear(8,1)` – multiple linear regression trained by gradient descent |
| **MLP** | one hidden layer (64, ReLU) |
| **Deep NN** | 128-64-32 ReLU with dropout |
| **F1+F2** | linear path + small non-linear (sigmoid) path, added together (class model) |
| **ResidualNet** | 2×64 ReLU layers + skip connection from a projected input (class model) |

In [ ]:
class ScaledRegressor(nn.Module):
    def __init__(self, x_means, x_deviations, y_means, y_deviations, log_age=True):
        super().__init__()
        self.log_age = log_age
        self.register_buffer("x_means", x_means.clone())
        self.register_buffer("x_deviations", x_deviations.clone())
        self.register_buffer("y_means", y_means.clone())
        self.register_buffer("y_deviations", y_deviations.clone())

    def core(self, x):
        raise NotImplementedError

    def forward(self, x):
        x = engineer(x, self.log_age)
        x = (x - self.x_means) / self.x_deviations
        pred_scaled = self.core(x)
        pred_descaled = pred_scaled * self.y_deviations + self.y_means
        return pred_descaled, pred_scaled

    def scale_y(self, y):
        return (y - self.y_means) / self.y_deviations


class LinearReg(ScaledRegressor):
    def __init__(self, *stats, n_in=8, **kw):
        super().__init__(*stats, **kw)
        self.linear1 = nn.Linear(n_in, 1)
    def core(self, x):
        return self.linear1(x)


class MLP(ScaledRegressor):
    def __init__(self, *stats, n_in=8, hidden=64, **kw):
        super().__init__(*stats, **kw)
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def core(self, x):
        return self.net(x)


class DeepNN(ScaledRegressor):
    def __init__(self, *stats, n_in=8, p=0.1, **kw):
        super().__init__(*stats, **kw)
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64),   nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, 32),    nn.ReLU(),
            nn.Linear(32, 1))
    def core(self, x):
        return self.net(x)


class F1plusF2(ScaledRegressor):
    """F1 (linear) + F2 (non-linear) – class model."""
    def __init__(self, *stats, n_in=8, hidden=10, **kw):
        super().__init__(*stats, **kw)
        self.f1_linear1 = nn.Linear(n_in, 1)
        self.f2_linear1 = nn.Linear(n_in, hidden)
        self.f2_act1    = nn.Sigmoid()
        self.f2_dropout = nn.Dropout(0.25)
        self.f2_linear2 = nn.Linear(hidden, 1)
    def core(self, x):
        f1 = self.f1_linear1(x)
        f2 = self.f2_linear2(self.f2_dropout(self.f2_act1(self.f2_linear1(x))))
        return f1 + f2


class ResidualNet(ScaledRegressor):
    """Residual MLP – class model."""
    def __init__(self, *stats, n_in=8, hidden=64, dropout_rate=0.1, **kw):
        super().__init__(*stats, **kw)
        self.fc1 = nn.Linear(n_in, hidden);   self.act1 = nn.ReLU(); self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden, hidden); self.act2 = nn.ReLU(); self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(hidden, 1)
        self.input_proj = nn.Linear(n_in, hidden)
    def core(self, x):
        x0 = self.input_proj(x)
        x1 = self.dropout1(self.act1(self.fc1(x)))
        x2 = self.dropout2(self.act2(self.fc2(x1)))
        return self.fc3(x2 + x0)


MODELS = {
    "Linear":      LinearReg,
    "MLP":         MLP,
    "Deep NN":     DeepNN,
    "F1+F2":       F1plusF2,
    "ResidualNet": ResidualNet,
}

## 5. Training loop (same for every model)
* Loss: MSE on the **scaled** target · Optimizer: Adam (lr = 1e-3, weight decay 1e-5) · batch = 32
* Up to 2000 epochs with **early stopping** on validation loss (patience 150); the best-validation weights are restored.
* Each architecture is trained with **5 random seeds** to measure stability (mean ± std).

In [ ]:
EPOCHS, PATIENCE, BATCH, LR = 2000, 150, 32, 1e-3
SEEDS = [0, 1, 2, 3, 4]

def predict(model, X_np):
    model.eval()
    with torch.no_grad():
        return model(to_t(X_np).to(device))[0].cpu().numpy()

def train_model(cls, seed, log_age=True, verbose=False):
    set_seed(seed)
    model = cls(*scaling_stats(log_age), log_age=log_age).to(device)
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = nn.MSELoss()
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, generator=g)
    Xv, yv = to_t(X_val).to(device), to_t(y_val).to(device)

    best_val, best_state, wait = float("inf"), None, 0
    hist = {"train": [], "val": []}
    for epoch in range(EPOCHS):
        model.train(); run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            _, pred_scaled = model(xb)
            loss = loss_fn(pred_scaled, model.scale_y(yb))
            loss.backward(); opt.step()
            run += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv)[1], model.scale_y(yv)).item()
        hist["train"].append(run / len(train_ds)); hist["val"].append(vl)
        if vl < best_val - 1e-6:
            best_val, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
        if verbose and epoch % 100 == 0:
            print(epoch, "train", round(hist["train"][-1], 4), "val", round(vl, 4))
    model.load_state_dict(best_state)
    return model, hist

def metrics(y_true, y_pred):
    return {"R2":   float(r2_score(y_true, y_pred)),
            "MAE":  float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred)))}

In [ ]:
rows, trained, histories = [], {}, {}
t0 = time.time()
for name, cls in MODELS.items():
    for seed in SEEDS:
        model, hist = train_model(cls, seed)
        mv, mt = metrics(y_val, predict(model, X_val)), metrics(y_test, predict(model, X_test))
        rows.append({"model": name, "seed": seed, "epochs": len(hist["val"]),
                     "val_R2": mv["R2"], "test_R2": mt["R2"], "test_MAE": mt["MAE"], "test_RMSE": mt["RMSE"]})
        trained[(name, seed)] = model; histories[(name, seed)] = hist
    r = [x for x in rows if x["model"] == name]
    print(f"{name:12s} test R2 = {np.mean([x['test_R2'] for x in r]):.4f} ± {np.std([x['test_R2'] for x in r]):.4f}"
          f"   ({time.time()-t0:.0f}s)")

runs = pd.DataFrame(rows)
runs.to_csv(f"{OUT_RES}/regression_all_runs.csv", index=False)

### Ablation: is the `log(1+age)` transform useful?
Same Linear and MLP models trained on raw age.

In [ ]:
abl = []
for name in ["Linear", "MLP"]:
    for log_age in [False, True]:
        r2s = []
        for seed in SEEDS:
            m, _ = train_model(MODELS[name], seed, log_age=log_age)
            r2s.append(r2_score(y_test, predict(m, X_test)))
        abl.append({"model": name, "age feature": "log(1+age)" if log_age else "raw age",
                    "test_R2_mean": float(np.mean(r2s)), "test_R2_std": float(np.std(r2s))})
ablation = pd.DataFrame(abl)
ablation.to_csv(f"{OUT_RES}/regression_ablation_logage.csv", index=False)
ablation

## 6. Results

In [ ]:
summary = (runs.groupby("model")
           .agg(val_R2=("val_R2", "mean"),
                test_R2_mean=("test_R2", "mean"), test_R2_std=("test_R2", "std"),
                test_MAE=("test_MAE", "mean"), test_RMSE=("test_RMSE", "mean"),
                epochs=("epochs", "mean"))
           .reindex(list(MODELS)))
summary.to_csv(f"{OUT_RES}/regression_summary.csv")
summary.round(4)

In [ ]:
# Model selection uses the VALIDATION set only; the test set is reported, not used to choose.
best_name = summary["val_R2"].idxmax()
best_seed = runs[runs.model == best_name].sort_values("val_R2", ascending=False).iloc[0]["seed"]
best_model = trained[(best_name, int(best_seed))]
y_pred = predict(best_model, X_test)
best_metrics = metrics(y_test, y_pred)
print("Best model (by validation R2):", best_name, "seed", int(best_seed))
print("Test metrics:", {k: round(v, 4) for k, v in best_metrics.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
s = summary.sort_values("test_R2_mean")
ax.barh(s.index, s["test_R2_mean"], xerr=s["test_R2_std"], color="#3b6ea5", capsize=4)
for i, v in enumerate(s["test_R2_mean"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center")
ax.set_xlim(max(0, s["test_R2_mean"].min() - 0.1), 1.0)
ax.set_xlabel("Test R² (mean ± std over 5 seeds)"); ax.set_title("Regression model comparison")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_model_comparison.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
h = histories[(best_name, int(best_seed))]
axes[0].plot(h["train"], label="train"); axes[0].plot(h["val"], label="validation")
axes[0].set_yscale("log"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE (scaled)")
axes[0].set_title(f"Learning curve – {best_name}"); axes[0].legend()

axes[1].scatter(y_test, y_pred, s=14, alpha=.7)
lim = [0, max(y_test.max(), y_pred.max()) + 5]
axes[1].plot(lim, lim, "k--", lw=1); axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_xlabel("actual strength (MPa)"); axes[1].set_ylabel("predicted (MPa)")
axes[1].set_title(f"Predicted vs actual (test) – R² = {best_metrics['R2']:.3f}")

res = (y_pred - y_test).ravel()
axes[2].scatter(y_pred, res, s=14, alpha=.7, color="#c0504d"); axes[2].axhline(0, color="k", lw=1)
axes[2].set_xlabel("predicted (MPa)"); axes[2].set_ylabel("residual (MPa)"); axes[2].set_title("Residuals (test)")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_best_model_diagnostics.png", dpi=150); plt.show()

### Feature importance (permutation importance on the test set, best model)
Shuffle one column at a time and measure how much R² drops.

In [ ]:
rng = np.random.default_rng(0)
base = r2_score(y_test, predict(best_model, X_test))
imp = {}
for j, f in enumerate(FEATURES):
    drops = []
    for _ in range(20):
        Xp = X_test.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
        drops.append(base - r2_score(y_test, predict(best_model, Xp)))
    imp[f] = float(np.mean(drops))
imp = pd.Series(imp).sort_values()
imp.plot.barh(figsize=(7, 4), color="#3b6ea5"); plt.xlabel("Drop in test R² when shuffled")
plt.title(f"Permutation importance – {best_name}")
plt.tight_layout(); plt.savefig(f"{OUT_FIG}/reg_permutation_importance.png", dpi=150); plt.show()
imp.sort_values(ascending=False)

## 7. Export the best model to ONNX (class approach) and verify it

In [ ]:
class ExportWrapper(nn.Module):
    """ONNX graph returns only the de-scaled prediction (MPa) as 'output1'."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x)[0]

export_model = ExportWrapper(copy.deepcopy(best_model).cpu()).eval().float()
dummy_input = torch.randn(1, len(FEATURES), dtype=torch.float32).abs() * 10 + 1
ONNX_PATH = f"{OUT_WEB}/concrete_model.onnx"
export_kw = dict(input_names=["input1"], output_names=["output1"], opset_version=15,
                 do_constant_folding=True,
                 dynamic_axes={"input1": {0: "batch"}, "output1": {0: "batch"}})
try:
    torch.onnx.export(export_model, dummy_input, ONNX_PATH, dynamo=False, **export_kw)
except TypeError:                     # older torch without the 'dynamo' argument
    torch.onnx.export(export_model, dummy_input, ONNX_PATH, **export_kw)
print("ONNX model saved:", ONNX_PATH, os.path.getsize(ONNX_PATH), "bytes")

In [ ]:
!pip -q install onnxruntime
import onnxruntime as ort
sess = ort.InferenceSession(ONNX_PATH)
onnx_pred = sess.run(["output1"], {"input1": X_test.astype(np.float32)})[0]
torch_pred = predict(best_model, X_test)
print("max |ONNX - PyTorch| =", float(np.abs(onnx_pred - torch_pred).max()))
print("ONNX test R2 =", r2_score(y_test, onnx_pred))
example = np.array([[540, 0, 0, 162, 2.5, 1040, 676, 28]], dtype=np.float32)
print("Example mix (row 1 of dataset, true 79.99 MPa) →", sess.run(None, {"input1": example})[0])

In [ ]:
# Metadata for the web page and the report
web_meta = {
    "task": "regression", "dataset": "Concrete Compressive Strength (UCI)",
    "features": FEATURES, "units": UNITS,
    "defaults": [float(v) for v in np.median(X_train, axis=0)],
    "mins": [float(v) for v in X.min(0)], "maxs": [float(v) for v in X.max(0)],
    "best_model": best_name, "best_seed": int(best_seed),
    "test_metrics": best_metrics,
    "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
    "n_rows_raw": int(len(df_raw)), "n_duplicates_removed": int(len(df_raw) - len(df)),
    "summary": summary.reset_index().to_dict(orient="records"),
    "ablation": ablation.to_dict(orient="records"),
    "permutation_importance": imp.sort_values(ascending=False).to_dict(),
    "target_corr": corr[TARGET].drop(TARGET).to_dict(),
    "torch_version": torch.__version__,
}
json.dump(web_meta, open(f"{OUT_WEB}/metrics.json", "w"), indent=2)
json.dump(web_meta, open(f"{OUT_RES}/regression_metrics.json", "w"), indent=2)
print("Saved metrics.json")

## 8. Analysis
The auto-generated report (`make_report.py`) turns the numbers above into the written analysis. Key points to look for:
* **Linear vs non-linear:** the Linear model is the baseline; the gap to MLP / Deep NN / ResidualNet shows how much of concrete strength is driven by non-linear interactions (e.g. water/cement ratio, ageing curve).
* **Feature engineering matters:** the ablation shows how much `log(1+age)` helps, especially for the Linear model.
* **Stability:** the std over 5 seeds tells whether a model's advantage is real or just a lucky initialization.
* **Residuals:** check whether errors grow for very high-strength mixes (fewer training examples there).
* **Deployment:** the ONNX model includes the age transform and both scalers, so the browser only feeds raw kg/m³ and days.

In [ ]:
# Zip everything for download (Colab)
import shutil
shutil.make_archive("regression_outputs", "zip", "outputs")
try:
    from google.colab import files; files.download("regression_outputs.zip")
except Exception:
    print("Saved regression_outputs.zip")